# 🤖 Üretken Yapay Zeka — Ders 1
## Üretken Modelleme Temelleri
### Likelihood · Posterior · Decision Theory · Generative Model

**Haydar Kılıç | Mühendislik Fakültesi, Yapay Zeka Mühendisliği**

---
Bu notebook, ders slaytlarındaki teorik kavramların Python ile uygulamalı gösterimini içermektedir.

**Gereksinimler:** Python 3.9+, NumPy 1.24+, Matplotlib 3.5+, SciPy 1.9+, scikit-learn 1.0+

In [ ]:
# Gerekli kütüphanelerin yüklenmesi
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.stats import norm
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# NumPy sürüm uyumluluğu: trapezoid, NumPy < 2.0'da trapz olarak geçer
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

# Grafik ayarları
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Kütüphaneler başarıyla yüklendi!')
print(f'   NumPy   : {np.__version__}')
import matplotlib; print(f'   Matplotlib: {matplotlib.__version__}')
import scipy;      print(f'   SciPy   : {scipy.__version__}')
import sklearn;    print(f'   sklearn : {sklearn.__version__}')

---
## 📌 BÖLÜM 1: Temel Kavramlar
### 1.1 El Yazısı Rakam Tanıma — Eğitim & Test Seti Kavramı

Her rakam **28×28 = 784** piksellik bir vektörle temsil edilir. 
Amacımız: $\mathbf{x}$ vektörünü alıp rakamın kimliğini $(0, \ldots, 9)$ döndüren bir fonksiyon öğrenmek.

Aşağıda bu fikri simüle ediyoruz:

In [ ]:
# 0-9 rakamlarını temsil eden sahte 28x28 görüntüler oluşturalım
np.random.seed(42)

def make_digit_image(digit, noise=0.15):
    """Basit bir rakam görüntüsü simüle eder."""
    # BUG FIX: np.random.seed() iç içe çağrısı global durumu bozuyordu.
    # Yeni NumPy Generator API kullanılarak izole rastgelelik sağlanır.
    rng = np.random.default_rng(digit * 7)
    img = np.zeros((28, 28))
    img[4:24, 4:24] = rng.random((20, 20)) * 0.3
    cx, cy = 14, 14
    for i in range(28):
        for j in range(28):
            d = ((i - cx)**2 + (j - cy)**2) ** 0.5
            if abs(d - (5 + digit % 4)) < 2:
                img[i, j] = 0.9
    img += rng.standard_normal((28, 28)) * noise
    img = np.clip(img, 0, 1)
    return img

# 10 rakamı görselleştir
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Eğitim Kümesinden Örnek El Yazısı Rakamlar (Simüle)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = make_digit_image(i)
    ax.imshow(img, cmap='Blues')
    ax.set_title(f'Rakam: {i}', fontsize=11)
    ax.axis('off')
    t = np.zeros(10, dtype=int)
    t[i] = 1
    # BUG FIX: np.array'i daha okunabilir biçimde göster
    ax.set_xlabel(f't = {list(t)}', fontsize=6)

plt.tight_layout()
plt.show()

print(f'\n📐 Vektör boyutu: 28 × 28 = {28*28} piksel')
print('🎯 Hedef: Her görüntüyü 0-9 arasında bir sınıfa atamak')

In [ ]:
# Eğitim / Test / Doğrulama ayrımı kavramsal gösterimi
np.random.seed(0)
N_total = 100
indices = np.arange(N_total)
np.random.shuffle(indices)

train_idx = indices[:70]
val_idx   = indices[70:85]
test_idx  = indices[85:]

fig, ax = plt.subplots(figsize=(12, 2))
ax.barh(0, 70, color='#2196F3', label='Eğitim Kümesi (70%)')
ax.barh(0, 15, left=70, color='#FF9800', label='Doğrulama Kümesi (15%)')
ax.barh(0, 15, left=85, color='#4CAF50', label='Test Kümesi (15%)')
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('Veri Oranı (%)')
ax.set_title('Veri Seti Bölümleme Stratejisi', fontweight='bold')
ax.legend(loc='lower right')

ax.text(35, 0, 'w katsayilarini ogretmek icin', ha='center', va='center', color='white', fontweight='bold')
ax.text(77.5, 0, 'M veya lambda\nsecimi icin', ha='center', va='center', color='white', fontsize=9)
ax.text(92.5, 0, 'Genelleme\nolcumu',          ha='center', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

---
### 1.2 Eğri Uydurma (Curve Fitting) ve Polinom Regresyon

Polinom fonksiyonu:
$$y(x, \mathbf{w}) = \sum_{j=0}^{M} w_j x^j$$

Minimize edilecek hata fonksiyonu (En Küçük Kareler):
$$E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^{N} \{y(x_n, \mathbf{w}) - t_n\}^2$$

In [ ]:
# Veri üret: sin(2πx) + gürültü
np.random.seed(42)
N = 10
x_train = np.linspace(0, 1, N)
t_train = np.sin(2 * np.pi * x_train) + np.random.normal(0, 0.3, N)

x_true = np.linspace(0, 1, 200)
t_true = np.sin(2 * np.pi * x_true)

def fit_polynomial(x, t, M):
    """M dereceli polinom uydurur, katsayıları döndürür.
    
    Çözüm: w* = (X^T X)^{-1} X^T t  (En Küçük Kareler)
    """
    X = np.vander(x, M + 1, increasing=True)  # Vandermonde matrisi
    w = np.linalg.lstsq(X, t, rcond=None)[0]
    return w

def poly_predict(x, w):
    """Polinom tahmin fonksiyonu."""
    M = len(w) - 1
    X = np.vander(x, M + 1, increasing=True)
    return X @ w

def rms_error(y_pred, t):
    """Ortalama Karekök Hatası: E_RMS = sqrt(mean((y-t)^2))"""
    return np.sqrt(np.mean((y_pred - t)**2))

# Farklı M dereceleri için görselleştirme
degrees = [0, 1, 3, 9]
colors  = ['#E91E63', '#9C27B0', '#2196F3', '#FF5722']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Polinom Eğri Uydurma — Farklı M Dereceleri', fontsize=15, fontweight='bold')

for ax, M, color in zip(axes.flat, degrees, colors):
    w = fit_polynomial(x_train, t_train, M)
    y_pred_train = poly_predict(x_train, w)
    y_pred_true  = poly_predict(x_true, w)

    E_train = rms_error(y_pred_train, t_train)

    ax.plot(x_true, t_true, 'g-', linewidth=2, label='Gerçek: sin(2πx)', zorder=1)
    ax.scatter(x_train, t_train, s=80, facecolors='none', edgecolors='blue',
               linewidths=2, zorder=3, label='Eğitim verisi')
    ax.plot(x_true, y_pred_true, color=color, linewidth=2.5,
            label=f'M={M} polinom', zorder=2)

    # Hata çizgilerini göster
    for xn, tn, yn in zip(x_train, t_train, y_pred_train):
        ax.plot([xn, xn], [tn, yn], 'g--', alpha=0.4, linewidth=1)

    ax.set_title(f'M = {M}  |  $E_{{RMS}}$ = {E_train:.4f}', fontsize=13)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.legend(fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('t')

plt.tight_layout()
plt.show()

---
### 1.3 Aşırı Uydurma (Overfitting) ve RMS Hata Karşılaştırması

$$E_{RMS} = \sqrt{\frac{2E(\mathbf{w}^*)}{N}}$$

M arttıkça eğitim hatası düşer ama test hatası yükselir → **Overfitting!**

In [ ]:
# Test verisi oluştur
np.random.seed(99)
N_test = 100
x_test = np.linspace(0, 1, N_test)
t_test = np.sin(2 * np.pi * x_test) + np.random.normal(0, 0.3, N_test)

# Farklı M değerleri için Train ve Test hatasını hesapla
max_degree = 9
train_errors = []
test_errors  = []

for M in range(max_degree + 1):
    w = fit_polynomial(x_train, t_train, M)

    y_train_pred = poly_predict(x_train, w)
    y_test_pred  = poly_predict(x_test, w)

    # Sayısal taşmayı önlemek için yüksek dereceli polinomlarda değerleri sınırla
    test_pred_clipped = np.clip(y_test_pred, -10, 10)

    train_errors.append(rms_error(y_train_pred, t_train))
    test_errors.append(rms_error(test_pred_clipped, t_test))

# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(max_degree + 1), train_errors, 'bo-', linewidth=2,
        markersize=8, label='Eğitim Hatası (Training)')
ax.plot(range(max_degree + 1), test_errors,  'ro-', linewidth=2,
        markersize=8, label='Test Hatası (Test)')

ax.axvline(x=3, color='green', linestyle='--', alpha=0.7, label='Optimal M=3')
ax.set_xlabel('Polinom Derecesi M', fontsize=13)
ax.set_ylabel('$E_{RMS}$', fontsize=13)
ax.set_title('Model Seçimi: Eğitim ve Test Hatası vs. Polinom Derecesi', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xticks(range(max_degree + 1))
ax.set_ylim(0, 1.2)

# Bölge açıklamaları
ax.axvspan(0, 2, alpha=0.08, color='red')
ax.axvspan(3, 4, alpha=0.08, color='green')
ax.axvspan(5, 9, alpha=0.08, color='orange')

ax.text(1,   1.1, 'Underfitting\n(Yetersiz Uyum)', ha='center', color='red',        fontsize=10)
ax.text(3.5, 1.1, 'İdeal',                         ha='center', color='green',      fontsize=10)
ax.text(7,   1.1, 'Overfitting\n(Aşırı Uyum)',     ha='center', color='darkorange', fontsize=10)

plt.tight_layout()
plt.show()

print('\n📊 Hata Tablosu:')
print(f'{"M":>3} | {"Train E_RMS":>12} | {"Test E_RMS":>12}')
print('-' * 33)
for M in range(max_degree + 1):
    marker = ' <- Optimal' if M == 3 else ''
    print(f'{M:>3} | {train_errors[M]:>12.4f} | {test_errors[M]:>12.4f}{marker}')

---
### 1.4 Düzenlileştirme (Regularization)

Aşırı uydurmanın önüne geçmek için hata fonksiyonuna ceza terimi ekliyoruz:

$$\tilde{E}(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^{N} \{y(x_n, \mathbf{w}) - t_n\}^2 + \frac{\lambda}{2} \|\mathbf{w}\|^2$$

$\lambda$ büyüdükçe model daha basit (daha küçük ağırlıklar) ama daha önyargılı hale gelir.

In [ ]:
def fit_polynomial_regularized(x, t, M, lam):
    """Ridge regression (L2 regularization) ile polinom uydurma.
    
    Çözüm: (X^T X + λI) w = X^T t
    """
    X = np.vander(x, M + 1, increasing=True)
    A = X.T @ X + lam * np.eye(M + 1)
    b = X.T @ t
    w = np.linalg.solve(A, b)
    return w

# M=9, farklı λ değerleri
M = 9
lambdas = {
    r'$\ln\lambda = -18\ (\lambda \approx 0)$': np.exp(-18),
    r'$\ln\lambda = -5$':                        np.exp(-5),
    r'$\ln\lambda = 0\ (\lambda = 1)$':          1.0,
    r'$\ln\lambda = 3$':                         np.exp(3),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle(r'Düzenlileştirme — M=9 Polinom, Farklı $\lambda$ Değerleri',
             fontsize=14, fontweight='bold')

for ax, (label, lam) in zip(axes.flat, lambdas.items()):
    w = fit_polynomial_regularized(x_train, t_train, M, lam)
    y_fit = poly_predict(x_true, w)
    y_fit = np.clip(y_fit, -2, 2)

    E_rms  = rms_error(poly_predict(x_train, w), t_train)
    w_norm = np.linalg.norm(w)

    ax.plot(x_true, t_true, 'g-', lw=2, label='Gerçek sin(2πx)')
    ax.scatter(x_train, t_train, s=80, facecolors='none', edgecolors='blue', lw=2)
    ax.plot(x_true, y_fit, 'r-', lw=2.5, label=label)
    ax.set_title(f'{label}\n$E_{{RMS}}={E_rms:.3f}$, $\\|\\mathbf{{w}}\\|={w_norm:.2f}$', fontsize=11)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.legend(fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('t')

plt.tight_layout()
plt.show()

print('💡 Anahtar Fikir:')
print('  λ → 0 : Düzenlileştirme yok  → Overfitting riski yüksek')
print('  λ → ∞ : Aşırı düzenlileştirme → Underfitting (model çok basit)')
print('  Optimal λ : Doğrulama kümesiyle belirlenir!')

---
## 📌 BÖLÜM 2: Olasılık Teorisi
### 2.1 Bileşik, Marjinal ve Koşullu Olasılık

- **Toplam Kuralı (Sum Rule):** $p(X) = \sum_Y p(X, Y)$
- **Çarpım Kuralı (Product Rule):** $p(X, Y) = p(Y|X)\, p(X)$
- **Bayes Teoremi:** $p(Y|X) = \dfrac{p(X|Y)\,p(Y)}{p(X)}$

In [ ]:
# Bileşik dağılım simülasyonu (derste gösterilen şekil)
np.random.seed(0)
N_samples = 60

# Y=1 ve Y=2 sınıfları
x_y1 = np.random.normal(2, 1.2, 35)   # Y=1: düşük X değerleri
x_y2 = np.random.normal(5, 1.2, 25)   # Y=2: yüksek X değerleri
y1_vals = np.ones(35)
y2_vals = 2 * np.ones(25)

x_all = np.concatenate([x_y1, x_y2])
y_all = np.concatenate([y1_vals, y2_vals])

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Bileşik, Marjinal ve Koşullu Olasılık Gösterimi', fontsize=14, fontweight='bold')

# Sol üst: p(X, Y) — bileşik dağılım
ax = axes[0, 0]
ax.scatter(x_y1, y1_vals, color='blue', s=50, alpha=0.7, label='Y=1')
ax.scatter(x_y2, y2_vals, color='blue', s=50, alpha=0.7, label='Y=2')
ax.set_yticks([1, 2])
ax.set_yticklabels(['Y=1', 'Y=2'])
ax.set_title('$p(X, Y)$ — Bileşik Dağılım', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Y')

# Sağ üst: p(Y) — marjinal dağılım
ax = axes[0, 1]
p_y1 = len(x_y1) / N_samples
p_y2 = len(x_y2) / N_samples
ax.barh([1, 2], [p_y1, p_y2], color=['#7986CB', '#5C6BC0'], height=0.5)
ax.set_title('$p(Y)$ — Marjinal Olasılık', fontweight='bold')
ax.set_xlabel('Olasılık'); ax.set_yticks([1, 2])
ax.set_yticklabels(['Y=1', 'Y=2'])
ax.set_xlim(0, 1)
for v, p in zip([1, 2], [p_y1, p_y2]):
    ax.text(p + 0.02, v, f'{p:.2f}', va='center', fontsize=12)

# Sol alt: p(X) — marjinal dağılım (toplam kuralı)
ax = axes[1, 0]
ax.hist(x_all, bins=15, color='#7986CB', edgecolor='white', density=True, alpha=0.8)
x_line = np.linspace(x_all.min() - 1, x_all.max() + 1, 200)
p_x_marginal = (p_y1 * norm.pdf(x_line, 2, 1.2) +
                p_y2 * norm.pdf(x_line, 5, 1.2))
ax.plot(x_line, p_x_marginal, 'r-', lw=2, label=r'$p(X) = \sum_Y p(X,Y)$')
ax.set_title('$p(X)$ — Marjinal Dağılım (Toplam Kuralı)', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Yoğunluk')
ax.legend()

# Sağ alt: p(X|Y=1) — koşullu dağılım
ax = axes[1, 1]
ax.hist(x_y1, bins=12, color='#EF9A9A', edgecolor='white', density=True,
        alpha=0.8, label='Y=1 gözlemleri')
x_line_y1 = np.linspace(x_y1.min() - 1, x_y1.max() + 1, 200)
ax.plot(x_line_y1, norm.pdf(x_line_y1, 2, 1.2), 'r-', lw=2, label='Fitted $p(X|Y=1)$')
ax.set_title('$p(X|Y=1)$ — Koşullu Dağılım', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Yoğunluk')
ax.legend()

plt.tight_layout()
plt.show()

print('\n📐 Toplam Kuralı Doğrulaması:')
print(f'  p(Y=1) = {p_y1:.4f},  p(Y=2) = {p_y2:.4f}')
print(f'  p(Y=1) + p(Y=2) = {p_y1 + p_y2:.4f} ≈ 1.0 ✅')

---
### 2.2 Bayes Teoremi — Hasta Teşhisi Örneği

$$p(Y|X) = \frac{p(X|Y)\,p(Y)}{p(X)}$$

**Gerçek Dünya Örneği:** Bir hastalık testi
- $C_1$: Hasta, $C_2$: Sağlıklı  
- Test pozitif geldi, gerçekten hasta olma olasılığı nedir?

In [ ]:
# Bayes Teoremi — Hasta Teşhisi
def bayes_diagnosis(p_disease, sensitivity, specificity):
    """
    Parametreler
    ------------
    p_disease   : P(C1) - hastalığın yaygınlığı (önsel / prior)
    sensitivity : P(test+ | C1) - gerçek pozitif oranı
    specificity : P(test- | C2) - gerçek negatif oranı

    Döndürür
    --------
    (P(C1|test+), P(C2|test+)) — sonsal olasılıklar
    """
    p_healthy           = 1 - p_disease
    p_pos_given_sick    = sensitivity
    p_pos_given_healthy = 1 - specificity

    # Toplam olasılık (normalizasyon): p(X=+)
    p_positive = (p_pos_given_sick    * p_disease +
                  p_pos_given_healthy * p_healthy)

    # Bayes teoremi: P(C1 | X=+)
    p_sick_given_pos    = (p_pos_given_sick    * p_disease) / p_positive
    p_healthy_given_pos = (p_pos_given_healthy * p_healthy) / p_positive

    return p_sick_given_pos, p_healthy_given_pos

# Parametreler
p_disease   = 0.01   # Hastalık yaygınlığı: %1
sensitivity = 0.95   # Duyarlılık: %95
specificity = 0.90   # Özgüllük: %90

p_sick_pos, p_healthy_pos = bayes_diagnosis(p_disease, sensitivity, specificity)

print('🏥 HASTA TEŞHISI — BAYES TEOREMI UYGULAMASI')
print('=' * 50)
print(f'\n📊 Önsel Bilgi (Prior):')
print(f'   P(Hasta)    = {p_disease:.2f}  (%{p_disease*100:.0f})')
print(f'   P(Sağlıklı) = {1-p_disease:.2f}  (%{(1-p_disease)*100:.0f})')
print(f'\n🔬 Test Performansı:')
print(f'   Duyarlılık (Sensitivity) = {sensitivity:.2f}')
print(f'   Özgüllük   (Specificity) = {specificity:.2f}')
print(f'\n✅ TEST POZİTİF GELDİ — Sonsal Olasılıklar (Posterior):')
print(f'   P(Hasta    | test+) = {p_sick_pos:.4f}  (%{p_sick_pos*100:.1f})')
print(f'   P(Sağlıklı | test+) = {p_healthy_pos:.4f}  (%{p_healthy_pos*100:.1f})')
print(f'\n💡 Sürpriz! Test pozitif olmasına rağmen gerçek hasta olma')
print(f'   olasılığı sadece %{p_sick_pos*100:.1f}!')
print(f'   Bu durum hastalığın az yaygın olmasından kaynaklanır (base rate fallacy).')

# Farklı yaygınlık değerleri için nasıl değiştiğini göster
prevalences = np.linspace(0.001, 0.5, 100)
posteriors  = [bayes_diagnosis(p, sensitivity, specificity)[0] for p in prevalences]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prevalences * 100, [p * 100 for p in posteriors], 'b-', lw=2.5)
ax.axvline(x=p_disease * 100, color='red', ls='--', alpha=0.7,
           label=f'Mevcut yaygınlık = %{p_disease*100:.0f}')
ax.axhline(y=p_sick_pos * 100, color='green', ls='--', alpha=0.7,
           label=f'Sonsal = %{p_sick_pos*100:.1f}')
ax.scatter([p_disease * 100], [p_sick_pos * 100], color='red', s=100, zorder=5)
ax.set_xlabel('Hastalık Yaygınlığı P(C₁) [%]', fontsize=12)
ax.set_ylabel('P(Hasta | Test+) [%]', fontsize=12)
ax.set_title('Bayes Teoremi: Yaygınlık vs. Sonsal Olasılık\n'
             '(Sensitivity=95%, Specificity=90%)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
### 2.3 Gaussian (Normal) Dağılım

$$\mathcal{N}(x|\mu, \sigma^2) = \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{-\frac{1}{2\sigma^2}(x-\mu)^2\right\}$$

**Beklenen değer:** $\mathbb{E}[x] = \mu$  
**Varyans:** $\text{var}[x] = \sigma^2$

In [ ]:
# Gaussian Dağılım Görselleştirmesi
x_range = np.linspace(-5, 8, 500)

params = [
    {'mu': 0, 'sigma2': 1,   'color': '#2196F3', 'label': r'$\mu=0,\ \sigma^2=1$'},
    {'mu': 2, 'sigma2': 0.5, 'color': '#E91E63', 'label': r'$\mu=2,\ \sigma^2=0.5$'},
    {'mu': 0, 'sigma2': 3,   'color': '#4CAF50', 'label': r'$\mu=0,\ \sigma^2=3$'},
    {'mu': 3, 'sigma2': 2,   'color': '#FF9800', 'label': r'$\mu=3,\ \sigma^2=2$'},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gaussian (Normal) Dağılım', fontsize=14, fontweight='bold')

# Sol: PDF
for p in params:
    y = norm.pdf(x_range, p['mu'], np.sqrt(p['sigma2']))
    axes[0].plot(x_range, y, color=p['color'], lw=2.5, label=p['label'])
    sigma = np.sqrt(p['sigma2'])
    axes[0].axvspan(p['mu'] - sigma, p['mu'] + sigma, alpha=0.05, color=p['color'])

axes[0].set_title('Olasılık Yoğunluk Fonksiyonu $p(x)$', fontsize=12)
axes[0].set_xlabel('x'); axes[0].set_ylabel('p(x)')
axes[0].legend(fontsize=10)

# Sağ: CDF
# BUG FIX: başlıkta raw string (r"") kullanılarak LaTeX \int doğru işlenir
for p in params:
    y_cdf = norm.cdf(x_range, p['mu'], np.sqrt(p['sigma2']))
    axes[1].plot(x_range, y_cdf, color=p['color'], lw=2.5, label=p['label'])

axes[1].set_title(r'Birikimli Dağılım Fonksiyonu $P(z)=\int_{-\infty}^{z}p(x)\,dx$', fontsize=12)
axes[1].set_xlabel('z'); axes[1].set_ylabel('P(z)')
axes[1].legend(fontsize=10)
axes[1].axhline(y=0.5, color='gray', ls=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Sayısal Doğrulama
print('🔢 Sayısal Doğrulama: E[x] ve Var[x]')
print('-' * 55)
rng_verify = np.random.default_rng(0)
for p in params:
    samples = rng_verify.normal(p['mu'], np.sqrt(p['sigma2']), 100_000)
    print(f"  mu={p['mu']}, sigma2={p['sigma2']}  "
          f"→  E[x]={samples.mean():.3f} (beklenen {p['mu']}), "
          f"Var[x]={samples.var():.3f} (beklenen {p['sigma2']})")

---
### 2.4 Maksimum Olabilirlik Tahmini (Maximum Likelihood Estimation)

Log-olabilirlik fonksiyonu:
$$\ln p(\mathbf{x}|\mu, \sigma^2) = -\frac{1}{2\sigma^2} \sum_{n=1}^{N}(x_n - \mu)^2 - \frac{N}{2}\ln\sigma^2 - \frac{N}{2}\ln(2\pi)$$

MLE çözümleri:
$$\mu_{ML} = \frac{1}{N}\sum_{n=1}^{N} x_n \qquad \sigma^2_{ML} = \frac{1}{N}\sum_{n=1}^{N}(x_n - \mu_{ML})^2$$

> ⚠️ $\sigma^2_{ML}$ **yanlı** (biased) bir tahmindir: gerçek varyansı küçük tahmin eder.

In [ ]:
# MLE Görselleştirmesi
np.random.seed(7)
mu_true    = 3.0
sigma_true = 1.5
N_mle = 50
x_obs = np.random.normal(mu_true, sigma_true, N_mle)

# MLE tahminleri
mu_mle          = np.mean(x_obs)
sigma2_mle      = np.var(x_obs)         # Yanlı (biased):   1/N
sigma2_unbiased = np.var(x_obs, ddof=1) # Yansız (unbiased): 1/(N-1)

print('📊 MLE TAHMİNLERİ')
print('=' * 45)
print(f'Gerçek değerler : μ = {mu_true:.2f},  σ² = {sigma_true**2:.2f}')
print(f'MLE tahminleri  : μ_ML = {mu_mle:.4f},  σ²_ML = {sigma2_mle:.4f}')
print(f'Yansız tahmin   : σ²_unbiased = {sigma2_unbiased:.4f}  (N/(N-1) × σ²_ML)')
# BUG FIX: 'YANLIDIR' sozcugunde Kiril harfi vardi -> ASCII 'R' ile duzeltildi
print(f'\n⚠️  MLE varyans tahmini YANLIDIR (biased)!')
print(f'   σ²_ML varyansı {sigma_true**2 - sigma2_mle:.4f} kadar küçük tahmin ediyor.')

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('MLE: Gaussian Parametre Tahmini', fontsize=14, fontweight='bold')

# Sol: Gözlemler ve fit edilen dağılım
ax = axes[0]
x_plot = np.linspace(x_obs.min() - 1, x_obs.max() + 1, 300)
ax.hist(x_obs, bins=12, density=True, alpha=0.5, color='skyblue',
        edgecolor='white', label='Gözlemler')
ax.plot(x_plot, norm.pdf(x_plot, mu_true, sigma_true), 'g-', lw=2.5,
        label=f'Gerçek: μ={mu_true}, σ={sigma_true}')
ax.plot(x_plot, norm.pdf(x_plot, mu_mle, np.sqrt(sigma2_mle)), 'r--', lw=2.5,
        label=f'MLE: μ={mu_mle:.2f}, σ={np.sqrt(sigma2_mle):.2f}')
ax.scatter(x_obs, np.zeros_like(x_obs), color='blue', s=20, alpha=0.5,
           zorder=5, label='Veri noktaları')
ax.set_xlabel('x'); ax.set_ylabel('Yoğunluk')
ax.set_title('Fit Edilen Gaussian Dağılım')
ax.legend(fontsize=9)

# Sağ: N arttıkça MLE yakınsama davranışı
ax = axes[1]
N_vals = np.arange(5, 501, 5)
mu_estimates     = []
sigma2_estimates = []
np.random.seed(42)
big_sample = np.random.normal(mu_true, sigma_true, 500)

for n in N_vals:
    sub = big_sample[:n]
    mu_estimates.append(np.mean(sub))
    sigma2_estimates.append(np.var(sub))

ax.plot(N_vals, mu_estimates,     'b-', lw=1.5, label='μ_ML tahmini')
ax.axhline(y=mu_true,            color='blue', ls='--', alpha=0.6,
           label=f'Gerçek μ={mu_true}')
ax.plot(N_vals, sigma2_estimates, 'r-', lw=1.5, label='σ²_ML tahmini')
ax.axhline(y=sigma_true**2,      color='red', ls='--', alpha=0.6,
           label=f'Gerçek σ²={sigma_true**2}')
ax.set_xlabel('Örnek Sayısı N')
ax.set_ylabel('Parametre Tahmini')
ax.set_title('N → ∞ iken MLE Yakınsaması')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
### 2.5 Olabilirlik, Önsel ve Sonsal Olasılık

$$\underbrace{p(\mathbf{w}|\mathcal{D})}_{\text{Sonsal}} = \frac{\underbrace{p(\mathcal{D}|\mathbf{w})}_{\text{Olabilirlik}} \cdot \underbrace{p(\mathbf{w})}_{\text{Önsel}}}{\underbrace{p(\mathcal{D})}_{\text{Normalizasyon}}}$$

**Sözlü:** sonsal ∝ olabilirlik × önsel

In [ ]:
# Bayesçi Güncelleme — Madeni Para Atışı Örneği
# θ = yazı gelme olasılığı; gözlemler: 1=yazı, 0=tura

theta = np.linspace(0, 1, 500)

# Önsel: düzgün (uniform) — herhangi bir ön bilgi yok
prior = np.ones_like(theta)
prior /= prior.sum()

observations = [1, 1, 0, 1, 1, 0, 1, 1, 1, 0]  # 7 yazı, 3 tura

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Bayesçi Güncelleme: Madeni Para Atışı\n(Önsel → Sonsal)',
             fontsize=14, fontweight='bold')

posterior = prior.copy()
heads, tails = 0, 0

for idx, (ax, obs) in enumerate(zip(axes.flat, observations)):
    likelihood = (theta ** obs) * ((1 - theta) ** (1 - obs))
    posterior  = posterior * likelihood
    posterior /= posterior.sum()  # normalize

    if obs == 1:
        heads += 1
    else:
        tails += 1

    map_estimate = theta[np.argmax(posterior)]
    mle_estimate = heads / (heads + tails)

    ax.fill_between(theta, posterior, alpha=0.4, color='#5C6BC0')
    ax.plot(theta, posterior, '#3949AB', lw=2)
    ax.axvline(x=map_estimate, color='red',   ls='--', lw=1.5, label=f'MAP={map_estimate:.2f}')
    ax.axvline(x=mle_estimate, color='green', ls=':',  lw=1.5, label=f'MLE={mle_estimate:.2f}')
    ax.axvline(x=0.6,          color='gray',  ls='-',  lw=1,   alpha=0.5, label='Gerçek=0.6')

    result = '✅ Yazı' if obs == 1 else '❌ Tura'
    ax.set_title(f'Atış {idx+1}: {result}\nY={heads}, T={tails}', fontsize=9)
    ax.set_xlabel(r'$\theta$', fontsize=9); ax.set_ylabel(r'$p(\theta|D)$', fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print('\n💡 Gözlem Sayısı Arttıkça:')
print('   → Önsel bilginin etkisi azalır')
print('   → Sonsal dağılım darlaşır (belirsizlik azalır)')
print('   → MAP tahmini MLE tahminine yaklaşır')

---
## 📌 BÖLÜM 3: Karar Teorisi (Decision Theory)
### 3.1 Sınıflandırmada Minimum Hata — Karar Sınırları

Minimum hata olasılığı için her x değerini **en yüksek sonsal olasılıklı** sınıfa ata:

$$\hat{C}(x) = \arg\max_k\ p(C_k|x)$$

In [ ]:
# Karar Sınırları Görselleştirmesi — 1D
x_plot = np.linspace(-2, 10, 500)

# İki sınıf parametreleri
mu1, sigma1, p_c1 = 2.5, 1.2, 0.45   # C1: Hasta
mu2, sigma2, p_c2 = 6.0, 1.5, 0.55   # C2: Sağlıklı

# Sınıf koşullu yoğunluklar
p_x_c1 = norm.pdf(x_plot, mu1, sigma1)
p_x_c2 = norm.pdf(x_plot, mu2, sigma2)

# Bileşik dağılımlar: p(x, Ck) = p(x|Ck) * p(Ck)
joint_c1 = p_x_c1 * p_c1
joint_c2 = p_x_c2 * p_c2

# Sonsal olasılıklar
p_x        = joint_c1 + joint_c2
posterior_c1 = joint_c1 / p_x
posterior_c2 = joint_c2 / p_x

# Karar sınırı: p(C1|x) = p(C2|x)
boundary_idx = np.argmin(np.abs(posterior_c1 - posterior_c2))
x0 = x_plot[boundary_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Karar Teorisi: Minimum Hata Karar Sınırı', fontsize=14, fontweight='bold')

# Sol: Bileşik dağılımlar
ax = axes[0]
ax.plot(x_plot, joint_c1, 'b-', lw=2.5, label='$p(x, C_1)$ - Hasta')
ax.plot(x_plot, joint_c2, 'r-', lw=2.5, label='$p(x, C_2)$ - Sağlıklı')
ax.axvline(x=x0, color='black', ls='--', lw=2, label=f'Karar sınırı $x_0={x0:.2f}$')

idx_left  = x_plot < x0
idx_right = x_plot > x0
ax.fill_between(x_plot[idx_left],  joint_c2[idx_left],  alpha=0.35, color='red',  label='Hata: $C_2 \\to R_1$')
ax.fill_between(x_plot[idx_right], joint_c1[idx_right], alpha=0.35, color='blue', label='Hata: $C_1 \\to R_2$')

ax.set_xlabel('x', fontsize=12); ax.set_ylabel('Olasılık', fontsize=12)
ax.set_title('Bileşik Dağılımlar $p(x, C_k)$')
ax.legend(fontsize=9)
# BUG FIX: Unicode ok '→' yerine LaTeX \rightarrow kullanıldı
ax.text(0.05, -0.12, r'$\mathcal{R}_1\ (\rightarrow C_1)$', transform=ax.transAxes,
        ha='left', fontsize=12)
ax.text(0.65, -0.12, r'$\mathcal{R}_2\ (\rightarrow C_2)$', transform=ax.transAxes,
        ha='left', fontsize=12)

# Sağ: Sonsal olasılıklar
ax = axes[1]
ax.plot(x_plot, posterior_c1, 'b-', lw=2.5, label='$p(C_1|x)$ - Hasta')
ax.plot(x_plot, posterior_c2, 'r-', lw=2.5, label='$p(C_2|x)$ - Sağlıklı')
ax.axvline(x=x0, color='black', ls='--', lw=2, label=f'$x_0 = {x0:.2f}$')
ax.axhline(y=0.5, color='gray', ls=':', alpha=0.5)
ax.fill_between(x_plot, 0, 1, where=(posterior_c1 > posterior_c2),
                alpha=0.1, color='blue', label=r'$\hat{C}=C_1$')
ax.fill_between(x_plot, 0, 1, where=(posterior_c2 > posterior_c1),
                alpha=0.1, color='red',  label=r'$\hat{C}=C_2$')
ax.set_xlabel('x', fontsize=12); ax.set_ylabel('Sonsal Olasılık', fontsize=12)
ax.set_title('Sonsal Olasılıklar $p(C_k|x)$')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# BUG FIX: np.trapz → np.trapezoid (NumPy 2.0'da kaldırıldı)
error_c2_in_r1 = np.trapezoid(joint_c2[idx_left],  x_plot[idx_left])
error_c1_in_r2 = np.trapezoid(joint_c1[idx_right], x_plot[idx_right])
total_error    = error_c2_in_r1 + error_c1_in_r2

print(f'\n📐 Karar Sınırı: x₀ = {x0:.3f}')
print(f'📊 Hata Analizi:')
print(f'   p(x ∈ R₁, C₂) = {error_c2_in_r1:.4f}  (C₂ → R₁ hatası)')
print(f'   p(x ∈ R₂, C₁) = {error_c1_in_r2:.4f}  (C₁ → R₂ hatası)')
print(f'   Toplam p(hata) = {total_error:.4f}')

---
### 3.2 Reddetme Seçeneği (Reject Option)

Sonsal olasılıkların belirsiz olduğu bölgelerde karar vermekten kaçınarak hata oranını düşürmek.

In [ ]:
# Reddetme Seçeneği Görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Reddetme Seçeneği (Reject Option)', fontsize=14, fontweight='bold')

thresholds    = [0.6, 0.8]
colors_reject = ['#FF9800', '#F44336']

for ax, theta_t, col in zip(axes, thresholds, colors_reject):
    ax.plot(x_plot, posterior_c1, 'b-', lw=2.5, label='$p(C_1|x)$')
    ax.plot(x_plot, posterior_c2, 'r-', lw=2.5, label='$p(C_2|x)$')
    ax.axhline(y=theta_t, color=col, ls='--', lw=2, label=f'Eşik θ={theta_t}')

    max_posterior = np.maximum(posterior_c1, posterior_c2)
    reject_mask   = max_posterior < theta_t
    accept_mask   = ~reject_mask

    ax.fill_between(x_plot, 0, 1, where=reject_mask,
                    alpha=0.25, color=col, label='Reddetme Bölgesi')
    ax.fill_between(x_plot, 0, 1, where=(accept_mask & (posterior_c1 > posterior_c2)),
                    alpha=0.1, color='blue', label=r'$\rightarrow C_1$')
    ax.fill_between(x_plot, 0, 1, where=(accept_mask & (posterior_c2 > posterior_c1)),
                    alpha=0.1, color='red',  label=r'$\rightarrow C_2$')

    rejected_pct = reject_mask.mean() * 100
    ax.set_title(f'θ = {theta_t}  |  Reddedilen: ~%{rejected_pct:.1f}', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('Sonsal Olasılık')
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('💡 Trade-off:')
print('   θ artarsa → daha fazla örnek reddedilir, kabul edilenlerde hata düşer')
print('   θ azalırsa → daha az örnek reddedilir, daha yüksek hata riski')

---
### 3.3 Kayıp Matrisi (Loss Matrix)

$$\mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} L_{kj}\, p(\mathbf{x}, C_k)\, d\mathbf{x}$$

Gerçek durum $C_k$ iken $C_j$ tahmin etmenin maliyeti $L_{kj}$.

In [ ]:
# Kayıp Matrisi Görselleştirmesi — Tıbbi Tanı Senaryosu
# Sınıflar: C1=Kanser, C2=Sağlıklı  |  Tahminler: R1=Kanser, R2=Sağlıklı
# L[gerçek, tahmin]
L_symmetric  = np.array([[0, 1],
                          [1, 0]])

L_asymmetric = np.array([[0, 10],   # Kanseri kaçırmak 10× daha maliyetli!
                          [1,  0]])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Kayıp Matrisi (Loss Matrix) Karşılaştırması', fontsize=14, fontweight='bold')

class_labels = ['$C_1$ (Kanser)', '$C_2$ (Sağlıklı)']
pred_labels  = ['$R_1$ (→Kanser)', '$R_2$ (→Sağlıklı)']

for ax, L, title in zip(axes,
                         [L_symmetric, L_asymmetric],
                         ['Simetrik Kayıp\n(Her hata eşit maliyetli)',
                          'Asimetrik Kayıp\n(Kanser kaçırmak 10× daha maliyetli)']):
    im = ax.imshow(L, cmap='YlOrRd', vmin=0, vmax=L.max())
    ax.set_xticks([0, 1]); ax.set_xticklabels(pred_labels, fontsize=11)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_labels, fontsize=11)
    ax.set_xlabel('Tahmin Edilen Sınıf', fontsize=11)
    ax.set_ylabel('Gerçek Sınıf', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

    for i in range(2):
        for j in range(2):
            color = 'white' if L[i, j] > L.max() / 2 else 'black'
            ax.text(j, i, f'{L[i, j]}', ha='center', va='center',
                    fontsize=20, fontweight='bold', color=color)
    plt.colorbar(im, ax=ax, label='Kayıp $L_{kj}$')

plt.tight_layout()
plt.show()

print('\n🔍 Asimetrik Kayıp Etkisi:')
print('   Simetrik : Her sınıfı eşit ağırlıkla yanlış sınıflandırmanın maliyeti aynı')
print('   Asimetrik: Kanseri "sağlıklı" olarak sınıflandırmak 10× daha maliyetli')
print('   → Asimetrik kayıpla karar sınırı C2 tarafına kayar (daha dikkatli tanı)')

---
### 3.4 Üretken, Ayrıştırıcı ve Diskriminant Modeller

| Model Tipi | Yaklaşım | Örnek |
|------------|----------|-------|
| **Üretken** | $p(x\|C_k)$ ve $p(C_k)$ → $p(C_k\|x)$ | Naive Bayes, GMM |
| **Ayrıştırıcı** | Doğrudan $p(C_k\|x)$ | Lojistik Regresyon |
| **Diskriminant** | $f(x)$ fonksiyonu | SVM, Perceptron |

In [ ]:
# 2D Sınıflandırma: Üç Yaklaşımın Karşılaştırması
np.random.seed(0)
X_cls, y_cls = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

models = [
    ('Üretken Model\n(Gaussian Naive Bayes)',   GaussianNB()),
    ('Ayrıştırıcı Model\n(Lojistik Regresyon)', LogisticRegression(random_state=0)),
    ('Diskriminant Fonksiyonu\n(SVM Doğrusal)', SVC(kernel='linear', probability=True, random_state=0)),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Üretken / Ayrıştırıcı / Diskriminant Model Karşılaştırması',
             fontsize=14, fontweight='bold')

colors_2d = ['#2196F3', '#E91E63']
x_min, x_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
y_min, y_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

for ax, (name, clf) in zip(axes, models):
    clf.fit(X_cls, y_cls)
    acc = clf.score(X_cls, y_cls)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=2, levels=[0.5])

    for cls_id, color in zip([0, 1], colors_2d):
        mask = y_cls == cls_id
        ax.scatter(X_cls[mask, 0], X_cls[mask, 1], c=color, s=40,
                   edgecolors='white', lw=0.5, label=f'$C_{cls_id+1}$')

    ax.set_title(f'{name}\nDoğruluk: %{acc*100:.1f}', fontsize=11)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print('\n📊 Üç Yaklaşımın Özeti:')
print('─' * 60)
print(f'{"Model":<30} {"Yaklaşım":<20} {"Olasılıksal?"}')
print('─' * 60)
print(f'{"Gaussian Naive Bayes":<30} {"Üretken":<20} Evet (tam bileşik)')
print(f'{"Lojistik Regresyon":<30} {"Ayrıştırıcı":<20} Evet (koşullu)')
print(f'{"SVM (Doğrusal)":<30} {"Diskriminant":<20} Hayır (skor tabanlı)')

---
## 📌 BÖLÜM 4: Özet ve Öğrenme Türleri

In [ ]:
# Makine Öğrenmesi Türleri — Özet Diyagramı
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Makine Öğrenmesi Yaklaşımları — Kapsamlı Özet',
             fontsize=16, fontweight='bold', pad=20)

boxes = [
    (0.3, 3.8, 2.8, 2.8, 'Gözetimli Öğrenme',
     'Etiketli veriler\n(x, t) çiftleri\n\n• Sınıflandırma\n• Regresyon\n\nÖrnek: Rakam tanıma',
     '#BBDEFB'),
    (3.6, 3.8, 2.8, 2.8, 'Gözetimsiz Öğrenme',
     'Etiketsiz veriler\nSadece x vektörleri\n\n• Kümeleme\n• Yoğunluk Kestirimi\n• Boyut İndirgeme',
     '#C8E6C9'),
    (6.9, 3.8, 2.8, 2.8, 'Pekiştirmeli Öğrenme',
     'Ödül maksimizasyonu\n\n• Keşif (Exploration)\n• İstifade (Exploitation)\n\nÖrnek: Oyun oynama',
     '#FFE0B2'),
    (0.3, 0.3, 2.8, 3.0, 'Olasılık Teorisi',
     '• Toplam Kuralı\n• Çarpım Kuralı\n• Bayes Teoremi\n• MLE\n• Gaussian Dağılım',
     '#E1BEE7'),
    (3.6, 0.3, 2.8, 3.0, 'Karar Teorisi',
     '• Karar Sınırları\n• Kayıp Matrisi\n• Min. Hata = Max. Posterior\n• Reddetme Seçeneği',
     '#FFCDD2'),
    (6.9, 0.3, 2.8, 3.0, 'Model Seçimi',
     '• Eğri Uydurma\n• Overfitting/Underfitting\n• Regularization (λ)\n• Validation Set',
     '#B2EBF2'),
]

for (x, y, w, h, title, content, color) in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.3, title,   ha='center', va='top',
            fontsize=11, fontweight='bold', color='#1A237E')
    ax.text(x + w/2, y + h - 0.7, content, ha='center', va='top',
            fontsize=9, color='#212121', linespacing=1.5)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Ders Özeti ───
print('=' * 60)
print('  📚 DERS 1 ÖZET — Anahtar Formüller')
print('=' * 60)

formulas = [
    ('Polinom Regresyon',
     'y(x,w) = Σ wⱼ xʲ  (j=0..M)'),
    ('Hata Fonksiyonu (En Küçük Kareler)',
     'E(w) = ½ Σ {y(xₙ,w) - tₙ}²'),
    ('RMS Hatası',
     'E_RMS = √(mean((y-t)²))'),
    ('Düzenlileştirme (Ridge)',
     'E~(w) = E(w) + (λ/2)||w||²'),
    ('Toplam Kuralı',
     'p(X) = Σ_Y p(X, Y)'),
    ('Çarpım Kuralı',
     'p(X, Y) = p(Y|X) p(X)'),
    ('Bayes Teoremi',
     'p(Y|X) = p(X|Y) p(Y) / p(X)'),
    ('Posterior ∝',
     'p(w|D) ∝ p(D|w) · p(w)'),
    ('Gaussian Dağılım',
     'N(x|μ,σ²) = (2πσ²)^(-½) exp{-(x-μ)²/(2σ²)}'),
    ('MLE — Ortalama',
     'μ_ML = (1/N) Σ xₙ'),
    ('MLE — Varyans (yanlı)',
     'σ²_ML = (1/N) Σ (xₙ - μ_ML)²'),
    ('Yansız Varyans',
     'σ²_unbiased = (1/(N-1)) Σ (xₙ - μ_ML)²'),
    ('Min. Hata Kuralı',
     'C_hat(x) = argmax_k p(Cₖ|x)'),
    ('Ortalama Kayıp',
     'E[L] = Σ_k Σ_j ∫_Rⱼ Lₖⱼ p(x,Cₖ) dx'),
]

for i, (name, formula) in enumerate(formulas, 1):
    print(f'\n  {i:>2}. {name}')
    print(f'      {formula}')

print('\n' + '=' * 60)
print('  ✅ Notebook tamamlandı!')
print('=' * 60)

---
## 🎯 Alıştırma Soruları

1. **Eğri Uydurma:** N=20 veri noktası için M=0,3,6,9 dereceli polinom uydurunuz. N=10 durumuyla kıyaslayınız. Overfitting davranışı değişiyor mu?

2. **Regularization:** λ parametresini 10⁻⁵'ten 10⁵'e kadar değiştirip her durumda hem eğitim hem test RMS hatasını hesaplayınız. "İdeal" λ nedir?

3. **Bayes Güncellemesi:** Madeni para örneğinde baskılı bir önsel dağılım (örn. Beta(5,5)) kullanınız. Uniform önselle karşılaştırınız.

4. **Kayıp Matrisi:** L₁₂=5 ve L₂₁=1 (yanlış negatif çok maliyetli) durumunda karar sınırı nasıl değişir?

5. **Model Karşılaştırma:** Farklı `class_sep` değerleri için Üretken, Ayrıştırıcı ve Diskriminant modellerin doğruluklarını karşılaştırınız.

---
*Üretken Yapay Zeka — Haydar Kılıç, Yapay Zeka Mühendisliği*